In [1]:
!pip install sqlalchemy pymysql pandas


In [13]:
# ========= IMPORTS =========
from __future__ import annotations
import os, datetime as dt
from pathlib import Path
from dataclasses import dataclass

import cv2
import numpy as np
from ultralytics import YOLO
from loguru import logger

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel

# DB
from sqlalchemy import create_engine, text, inspect

In [3]:
DB_USER = "root"
DB_PASS = "1"
DB_HOST = "127.0.0.1"      # hoặc IP/Docker service name
DB_PORT = 3306
DB_NAME = "behavior_detect"

url = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
engine = create_engine(url, pool_pre_ping=True)


In [10]:
SQL_CREATE_EVENTS = text("""
CREATE TABLE IF NOT EXISTS events (
    id             BIGINT AUTO_INCREMENT PRIMARY KEY,
    video_name     VARCHAR(255) NOT NULL,
    frame_idx      INT NOT NULL,
    timestamp_sec  DOUBLE NOT NULL,
    timestamp_hms  VARCHAR(32) NOT NULL,
    event_type     ENUM('smoke','drink') NOT NULL,
    obj_name       VARCHAR(64),
    score          FLOAT,
    created_at     TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    UNIQUE KEY uq_video_frame (video_name, frame_idx, event_type, obj_name),
    KEY idx_video_ts (video_name, timestamp_sec)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")

In [14]:
def init_db():
    insp = inspect(engine)
    if not insp.has_table("events", schema=DB_NAME):
        with engine.begin() as conn:
            conn.execute(SQL_CREATE_EVENTS)

In [15]:
init_db()

In [16]:
SQL_INSERT = text("""
INSERT IGNORE INTO events
(video_name, frame_idx, timestamp_sec, timestamp_hms, event_type, obj_name, score)
VALUES (:video_name, :frame_idx, :timestamp_sec, :timestamp_hms, :event_type, :obj_name, :score)
""")

In [5]:
def ensure_dir(path: str | Path):
    Path(path).mkdir(parents=True, exist_ok=True)

def save_temp_upload(upload_file, tmp_dir="/tmp") -> str:
    ensure_dir(tmp_dir)
    dest = Path(tmp_dir) / upload_file.filename
    with open(dest, "wb") as f:
        f.write(upload_file.file.read())
    return str(dest)

def video_writer(out_path: str, fps: float, w: int, h: int):
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    return cv2.VideoWriter(out_path, fourcc, fps, (w, h))

In [6]:
@dataclass
class Event:
    frame_idx: int
    timestamp: float
    timestamp_hms: str
    type: str                 # "smoke" | "drink"
    obj_name: str | None
    score: float

@dataclass
class DetectionConfig:
    general_model_path: str
    smoke_model_path: str
    imgsz: int = 640
    conf_thres: float = 0.25
    drink_objs: list[str] = None
    frame_skip: int = 3
    output_dir: str = "outputs"

class BehaviorDetector:
    def __init__(self, cfg: DetectionConfig):
        self.cfg = cfg
        logger.info("Loading models...")
        self.general_model = YOLO(cfg.general_model_path)
        self.smoke_model   = YOLO(cfg.smoke_model_path)
        ensure_dir(cfg.output_dir)

    def process(self, video_path: str, save_overlay: bool = True):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video: {video_path}")

        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        events: list[Event] = []
        out_path = None
        writer = None
        if save_overlay:
            out_path = str(Path(self.cfg.output_dir) / (Path(video_path).stem + "_overlay.mp4"))
            writer = video_writer(out_path, fps, w, h)

        frame_idx = 0
        drink_objs = set(self.cfg.drink_objs or [])

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % self.cfg.frame_skip != 0:
                frame_idx += 1
                continue

            ts_sec = frame_idx / fps
            ts_hms = str(dt.timedelta(seconds=ts_sec))

            # Detect general objects
            result_general = self.general_model.predict(
                source=frame, imgsz=self.cfg.imgsz, conf=self.cfg.conf_thres, verbose=False
            )[0]
            boxes_g = result_general.boxes.xyxy.cpu().numpy() if result_general.boxes is not None else np.zeros((0,4))
            cls_g   = result_general.boxes.cls.cpu().numpy().astype(int) if result_general.boxes is not None else np.zeros((0,))
            conf_g  = result_general.boxes.conf.cpu().numpy() if result_general.boxes is not None else np.zeros((0,))
            names_g = [result_general.names[i] for i in cls_g]

            # Detect cigarette
            result_smoke = self.smoke_model.predict(
                source=frame, imgsz=self.cfg.imgsz, conf=self.cfg.conf_thres, verbose=False
            )[0]
            boxes_s = result_smoke.boxes.xyxy.cpu().numpy() if result_smoke.boxes is not None else np.zeros((0,4))
            conf_s  = result_smoke.boxes.conf.cpu().numpy() if result_smoke.boxes is not None else np.zeros((0,))

            if len(boxes_s) > 0:
                best_i = int(np.argmax(conf_s))
                events.append(Event(frame_idx, ts_sec, ts_hms, "smoke", "cigarette", float(conf_s[best_i])))

            for b, n, sc in zip(boxes_g, names_g, conf_g):
                if n in drink_objs:
                    events.append(Event(frame_idx, ts_sec, ts_hms, "drink", n, float(sc)))

            if writer is not None:
                overlay = frame.copy()
                # draw general
                for (x1,y1,x2,y2), n, sc in zip(boxes_g, names_g, conf_g):
                    cv2.rectangle(overlay, (int(x1),int(y1)), (int(x2),int(y2)), (0,255,0), 2)
                    cv2.putText(overlay, f"{n}:{sc:.2f}", (int(x1), int(y1)-5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,0), 1)
                # draw smoke
                for (x1,y1,x2,y2), sc in zip(boxes_s, conf_s):
                    cv2.rectangle(overlay, (int(x1),int(y1)), (int(x2),int(y2)), (0,0,255), 2)
                    cv2.putText(overlay, f"cig:{sc:.2f}", (int(x1), int(y1)-5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,0,255), 1)
                writer.write(overlay)

            frame_idx += 1

        cap.release()
        if writer is not None:
            writer.release()

        events_json = [e.__dict__ for e in events]
        return {"events": events_json, "overlay_video": out_path}

In [7]:
app = FastAPI(title="Smoking/Drinking Behavior API", version="0.1.0")

# Load YAML config for model paths
import yaml
CFG_PATH = os.getenv("CFG_PATH", "config.yaml")
with open(CFG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

detector = BehaviorDetector(
    DetectionConfig(
        general_model_path=cfg["GENERAL_MODEL"],
        smoke_model_path=cfg["SMOKE_MODEL"],
        imgsz=cfg.get("IMGSZ", 640),
        conf_thres=cfg.get("CONF_THRES", 0.25),
        drink_objs=cfg.get("DRINK_OBJS", []),
        frame_skip=cfg.get("FRAME_SKIP", 3),
        output_dir=cfg.get("OUTPUT_DIR", "outputs"),
    )
)
ensure_dir(detector.cfg.output_dir)

class PredictResponse(BaseModel):
    events: list[dict]
    overlay_video: str | None

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
async def predict(file: UploadFile = File(...)):
    # Lưu file tạm
    try:
        tmp_path = save_temp_upload(file, tmp_dir="tmp")   # dùng thư mục local trên Windows
    except Exception as e:
        logger.exception(e)
        raise HTTPException(500, f"SAVE_FILE_ERROR: {e}")

    # Detect
    try:
        result = detector.process(tmp_path, save_overlay=True)
    except Exception as e:
        logger.exception(e)
        raise HTTPException(500, f"DETECTOR_ERROR: {e}")

    # Insert DB
    try:
        if result["events"]:
            rows = [{
                "video_name": Path(tmp_path).stem,
                "frame_idx": int(e["frame_idx"]),
                "timestamp_sec": float(e["timestamp"]),
                "timestamp_hms": str(e["timestamp_hms"]),
                "event_type": str(e["type"]),
                "obj_name": (e["obj_name"] or None),
                "score": float(e["score"]),
            } for e in result["events"]]

            with engine.begin() as conn:
                conn.execute(SQL_INSERT, rows)
    except Exception as e:
        logger.exception(e)
        raise HTTPException(500, f"DB_ERROR: {e}")

    # Build response
    try:
        return PredictResponse(**result)
    except Exception as e:
        logger.exception(e)
        raise HTTPException(500, f"RESPONSE_ERROR: {e}")

@app.get("/download")
def download(path: str):
    p = Path(path)
    if not p.exists():
        raise HTTPException(status_code=404, detail="File not found")
    media = "video/mp4" if p.suffix.lower() == ".mp4" else "application/octet-stream"
    return FileResponse(path, media_type=media, filename=p.name)

2025-07-23 18:07:40.884 | INFO     | __main__:__init__:23 - Loading models...


In [ ]:
import threading, uvicorn

def run_app():
    uvicorn.run(app, host="0.0.0.0", port=8000)

if "server_thread" not in globals():   # tránh start lại
    server_thread = threading.Thread(target=run_app, daemon=True)
    server_thread.start()
    print("Server running at http://127.0.0.1:8000")
else:
    print("Server already running.")


Server running at http://127.0.0.1:8000


INFO:     Started server process [37924]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52267 - "GET / HTTP/1.1" 307 Temporary Redirect
INFO:     127.0.0.1:52267 - "GET /ui/index.html HTTP/1.1" 200 OK
INFO:     127.0.0.1:52269 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:52304 - "POST /predict HTTP/1.1" 200 OK


In [9]:
from pathlib import Path
from fastapi.staticfiles import StaticFiles
from fastapi.responses import RedirectResponse

STATIC_DIR = Path.cwd() / "static"   # hoặc đường dẫn bạn muốn
STATIC_DIR.mkdir(exist_ok=True)      # tạo nếu chưa có

app.mount("/ui", StaticFiles(directory=str(STATIC_DIR), html=True), name="ui")

@app.get("/")
def root():
    return RedirectResponse("/ui/index.html")
